# 03 — Training: YOLOv26 Segmentation

**Goal**: Train a YOLOv26-seg model on the preprocessed dataset.

**Model recommendation by dataset size**:
| Dataset size | Recommended model | Notes |
|---|---|---|
| < 500 images | `yolo26s-seg.pt` | Best balance — used here |
| 500–2000 | `yolo26m-seg.pt` | If you augment aggressively |
| > 2000 | `yolo26l-seg.pt` / `yolo26x-seg.pt` | Larger capacity justified |

With ~294 training images (after 80/20 split from 367 matched pairs), **Small** is the right choice.

> **Prerequisite**: Run `02_preprocessing.ipynb` first — `dataset/` must exist.

## 1. GPU & Environment Verification

In [1]:
import torch
from ultralytics import YOLO
from pathlib import Path
import yaml, json
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import warnings
warnings.filterwarnings("ignore")

BASE_DIR = Path("..")  # project root

print("=" * 50)
print(f"PyTorch version   : {torch.__version__}")
print(f"CUDA available    : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU               : {gpu.name}")
    print(f"VRAM              : {gpu.total_memory / 1e9:.1f} GB")
    print(f"CUDA version      : {torch.version.cuda}")
    DEVICE = 0
else:
    print("⚠️  No GPU detected — training on CPU (much slower)")
    DEVICE = "cpu"

# Verify dataset exists
DATASET_YAML = BASE_DIR / "dataset" / "dataset.yaml"
assert DATASET_YAML.exists(), "Run 02_preprocessing.ipynb first"

with open(DATASET_YAML) as f:
    ds = yaml.safe_load(f)

print(f"\nDataset YAML      : {DATASET_YAML}")
print(f"Classes ({ds['nc']})     : {ds['names']}")
print("=" * 50)

PyTorch version   : 2.9.1+cu126
CUDA available    : True
GPU               : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM              : 8.6 GB
CUDA version      : 12.6

Dataset YAML      : ..\dataset\dataset.yaml
Classes (11)     : ['class_1', 'class_2', 'class_3', 'class_4', 'class_5', 'class_6', 'class_7', 'class_8', 'class_9', 'class_10', 'class_11']


## 2. Batch Size Guide

Set `BATCH_SIZE` based on your GPU VRAM:

| VRAM | Batch size (imgsz=640) |
|---|---|
| 4 GB | 4 |
| 6 GB | 8 |
| 8 GB | 12 |
| 10–12 GB | 16 |
| 16 GB+ | 24–32 |

Use `-1` to let Ultralytics auto-detect the maximum safe batch size.

In [2]:
# ── Training Configuration ──────────────────────────────────────────────────
MODEL_SIZE  = "s"      # n / s / m / l / x  (s = Small, recommended)
BATCH_SIZE  = -1       # -1 = auto; or set manually (see table above)
IMGSZ       = 640      # standard YOLO input size; 480 also fine (native resolution)
EPOCHS      = 100      # increase to 150–200 if val mAP still improving at 100
PATIENCE    = 25       # early stopping — stop if no improvement for N epochs
WORKERS     = 4        # data loading threads; 0 if you get DataLoader errors on Windows
SEED        = 42
PROJECT_DIR = str(BASE_DIR / "runs" / "segment")
RUN_NAME    = "yolo26s_run1"

print(f"Model   : yolo26{MODEL_SIZE}-seg.pt")
print(f"Batch   : {BATCH_SIZE}")
print(f"imgsz   : {IMGSZ}")
print(f"Epochs  : {EPOCHS}  (early stop patience={PATIENCE})")
print(f"Device  : {DEVICE}")
print(f"Output  : {PROJECT_DIR}/{RUN_NAME}")

Model   : yolo26s-seg.pt
Batch   : -1
imgsz   : 640
Epochs  : 100  (early stop patience=25)
Device  : 0
Output  : ..\runs\segment/yolo26s_run1


## 3. Load Pretrained YOLOv26 Model
Downloading pretrained weights transfers learned feature representations from COCO,
which is critical for small datasets like ours.

In [3]:
model = YOLO(f"yolo26{MODEL_SIZE}-seg.pt")
print(f"\nModel loaded: yolo26{MODEL_SIZE}-seg")
print("Architecture summary:")
model.info()


Model loaded: yolo26s-seg
Architecture summary:
YOLO26s-seg summary: 309 layers, 11,505,800 parameters, 0 gradients, 37.4 GFLOPs


(309, 11505800, 0, 37.42105600000001)

## 4. Train

### Key hyperparameters explained:
| Param | Value | Why |
|---|---|---|
| `amp=True` | True | Mixed precision — ~2× faster on GPU, same accuracy |
| `cos_lr=True` | True | Cosine LR decay — smoother convergence |
| `warmup_epochs=3` | 3 | Prevents large early gradient steps |
| `mosaic=0.8` | 0.8 | Mosaic augmentation — helps with small objects |
| `mixup=0.1` | 0.1 | Light MixUp — regularises with small data |
| `copy_paste=0.1` | 0.1 | Copy-paste augmentation — useful for segmentation |
| `overlap_mask=True`| True | Allow overlapping instance masks |
| `mask_ratio=4` | 4 | Downsampling ratio for proto masks |

> Training logs and weights are saved to `runs/segment/yolo26s_run1/`

In [4]:
results = model.train(
    # ── Data ────────────────────────────────────────────────
    data        = str(DATASET_YAML),
    imgsz       = IMGSZ,

    # ── Training schedule ───────────────────────────────────
    epochs      = EPOCHS,
    patience    = PATIENCE,
    batch       = BATCH_SIZE,
    device      = DEVICE,
    workers     = WORKERS,
    seed        = SEED,

    # ── Optimizer ───────────────────────────────────────────
    optimizer   = "AdamW",
    lr0         = 1e-3,    # initial learning rate
    lrf         = 0.01,    # final lr = lr0 × lrf (cosine decay endpoint)
    momentum    = 0.937,
    weight_decay= 5e-4,
    warmup_epochs    = 3,
    warmup_momentum  = 0.8,
    warmup_bias_lr   = 0.1,

    # ── Precision ───────────────────────────────────────────
    amp         = True,    # automatic mixed precision (FP16)
    cos_lr      = True,    # cosine learning rate schedule

    # ── Augmentation ────────────────────────────────────────
    hsv_h       = 0.015,   # hue jitter
    hsv_s       = 0.7,     # saturation jitter
    hsv_v       = 0.4,     # value (brightness) jitter
    degrees     = 5.0,     # rotation ±5°
    translate   = 0.1,
    scale       = 0.5,
    shear       = 2.0,
    perspective = 0.0,
    flipud      = 0.1,
    fliplr      = 0.5,
    mosaic      = 0.8,
    mixup       = 0.1,
    copy_paste  = 0.1,     # segmentation-specific augmentation

    # ── Segmentation ────────────────────────────────────────
    overlap_mask = True,
    mask_ratio   = 4,

    # ── Logging ─────────────────────────────────────────────
    project     = PROJECT_DIR,
    name        = RUN_NAME,
    save        = True,
    save_period = 10,      # checkpoint every N epochs
    plots       = True,    # save training curve plots
    verbose     = True,
)

print("\n✓ Training complete")
best_weights = Path(PROJECT_DIR) / RUN_NAME / "weights" / "best.pt"
print(f"Best weights: {best_weights}")

Ultralytics 8.4.41  Python-3.12.10 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=..\dataset\dataset.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo26s-seg.pt, momentum=0.937, mosaic=0.8, multi_scale=0.0, name=yolo26s_run1, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overl

2026/04/24 20:21:36 INFO mlflow.tracking.fluent: Experiment with name '..\runs\segment' does not exist. Creating a new experiment.


MLflow: logging run_id(1cd03152d45847ee924ee184effd6a1f) to runs\mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs\mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 4 dataloader workers
Logging results to C:\NaufalFirdaus\CODES\MyGithub\self-driving-car-yolov26\notebooks\runs\runs\segment\yolo26s_run1
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
      1/100      3.77G      2.051      3.607      3.478    0.01286      4.296         82        640: 100% ━━━━━━━━━━━━ 74/74 3.0it/s 24.4s0.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.4it/s 7.3s.2s
                   all         74       1956      0.446      0.198      0.179      0.103      0.416      0.156      0.125     0.0668

      Epoch    GPU_mem 

## 5. Training Curves
Review the plots to check for:
- Underfitting: training loss still decreasing at epoch 100 → increase epochs
- Overfitting: val loss diverges from train loss → add more augmentation or reduce model size

In [5]:
results_dir = Path(PROJECT_DIR) / RUN_NAME

# Ultralytics saves results.png with all training curves
results_png = results_dir / "results.png"
if results_png.exists():
    img = mpimg.imread(str(results_png))
    plt.figure(figsize=(18, 10))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Training Curves", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print(f"results.png not found at {results_png}")

results.png not found at ..\runs\segment\yolo26s_run1\results.png


In [6]:
# Show confusion matrix if available
conf_matrix = results_dir / "confusion_matrix_normalized.png"
if conf_matrix.exists():
    img = mpimg.imread(str(conf_matrix))
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Confusion Matrix (Normalized)", fontsize=13)
    plt.tight_layout()
    plt.show()

# Print final training metrics
print("\n=== Best Model Metrics (on validation set) ===")
print(f"Box mAP50    : {results.results_dict.get('metrics/mAP50(B)', 'N/A'):.4f}")
print(f"Box mAP50-95 : {results.results_dict.get('metrics/mAP50-95(B)', 'N/A'):.4f}")
print(f"Seg mAP50    : {results.results_dict.get('metrics/mAP50(M)', 'N/A'):.4f}")
print(f"Seg mAP50-95 : {results.results_dict.get('metrics/mAP50-95(M)', 'N/A'):.4f}")
print(f"\nBest weights saved at: {best_weights}")
print("\n✓ Proceed to 04_evaluation.ipynb")


=== Best Model Metrics (on validation set) ===
Box mAP50    : 0.7413
Box mAP50-95 : 0.4977
Seg mAP50    : 0.7204
Seg mAP50-95 : 0.4530

Best weights saved at: ..\runs\segment\yolo26s_run1\weights\best.pt

✓ Proceed to 04_evaluation.ipynb
